# KEK → PLN Grid Region Mapping

Maps each of the 25 KEKs to their PLN interconnected grid system for use in
GEAS allocation and RUPTL join.

**Output:** `data/kek_grid_region_mapping.csv`

**Grid regions used (PLN RUPTL 2021–2030 system names):**
| grid_region_id | Coverage |
|----------------|----------|
| JAVA_BALI | Java + Bali interconnected system |
| SUMATRA | Sumatra interconnected system (incl. Bangka-Belitung, Riau Islands) |
| KALIMANTAN_SOUTH_CENTRAL | South + Central Kalimantan system |
| KALIMANTAN_EAST | East + North Kalimantan system |
| SULAWESI_SOUTH | South + Southeast Sulawesi system |
| SULAWESI_NORTH_CENTRAL | North + Central Sulawesi system |
| MALUKU_PAPUA | Maluku + Papua + West Papua |
| NUSA_TENGGARA | NTB + NTT |

**Source:** Province/address from `kek_info_and_markers.csv` + lat/lon cross-check.
Review the map below and adjust any assignments before saving.

In [ ]:
from pathlib import Path

import folium
import pandas as pd

RAW = Path("../outputs/data/raw")
DATA = Path("../data")

kek = pd.read_csv(
    RAW / "kek_info_and_markers.csv",
    usecols=["xid", "slug", "title", "address", "latitude", "longitude"],
)
print(f"{len(kek)} KEKs loaded")
kek[["xid", "title", "address", "latitude", "longitude"]]

## Assign grid regions

Based on province from address field + lat/lon cross-check.
**Edit the mapping below if any assignment looks wrong.**

In [ ]:
# slug → grid_region_id
# Derived from province in address field; verify against map below.
GRID_MAP = {
    # Java + Bali
    "industropolis-batang": "JAVA_BALI",  # Central Java
    "bumi-serpong-damai": "JAVA_BALI",  # Banten
    "kek-sanur": "JAVA_BALI",  # Bali
    "kek-kendal": "JAVA_BALI",  # Central Java
    "kek-singhasari": "JAVA_BALI",  # East Java
    "kek-gresik": "JAVA_BALI",  # East Java
    "kek-kura-kura": "JAVA_BALI",  # Bali
    "kek-lido": "JAVA_BALI",  # West Java
    "kek-tanjung-lesung": "JAVA_BALI",  # Banten
    # Sumatra
    "batam-tourism-and-international-healthcare": "SUMATRA",  # Riau Islands (linked to Sumatra)
    "tanjung-sauh": "SUMATRA",  # Batam, Riau Islands
    "kek-arun-lhokseumawe": "SUMATRA",  # Aceh
    "kek-galang-batang": "SUMATRA",  # Bintan, Riau Islands
    "kek-nongsa": "SUMATRA",  # Batam, Riau Islands
    "batam-aero-technic": "SUMATRA",  # Batam, Riau Islands
    "kek-tanjung-kelayang": "SUMATRA",  # Belitung, Bangka-Belitung
    "kek-sei-mangkei": "SUMATRA",  # North Sumatra
    # Kalimantan
    "setangga": "KALIMANTAN_SOUTH_CENTRAL",  # South Kalimantan
    "kek-maloy-batuta": "KALIMANTAN_EAST",  # East Kalimantan
    # Sulawesi
    "kek-likupang": "SULAWESI_NORTH_CENTRAL",  # North Sulawesi
    "kek-bitung": "SULAWESI_NORTH_CENTRAL",  # North Sulawesi
    "kek-palu": "SULAWESI_NORTH_CENTRAL",  # Central Sulawesi
    # Maluku + Papua
    "kek-sorong": "MALUKU_PAPUA",  # West Papua
    "kek-morotai": "MALUKU_PAPUA",  # North Maluku
    # Nusa Tenggara
    "kek-mandalika": "NUSA_TENGGARA",  # West Nusa Tenggara (Lombok)
}

assert len(GRID_MAP) == len(kek), f"Missing slugs: {set(kek.slug) - set(GRID_MAP)}"

kek["grid_region_id"] = kek["slug"].map(GRID_MAP)
print(kek.groupby("grid_region_id")["title"].count().sort_values(ascending=False))
kek[["slug", "title", "address", "grid_region_id"]]

## Visual check — map with grid region colours

In [ ]:
REGION_COLORS = {
    "JAVA_BALI": "blue",
    "SUMATRA": "green",
    "KALIMANTAN_SOUTH_CENTRAL": "orange",
    "KALIMANTAN_EAST": "darkred",
    "SULAWESI_NORTH_CENTRAL": "purple",
    "SULAWESI_SOUTH": "darkpurple",
    "MALUKU_PAPUA": "red",
    "NUSA_TENGGARA": "cadetblue",
}

m = folium.Map(location=[-2, 118], zoom_start=5, tiles="CartoDB positron")

for _, row in kek.iterrows():
    color = REGION_COLORS.get(row["grid_region_id"], "gray")
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=7,
        color=color,
        fill=True,
        fill_opacity=0.9,
        tooltip=f"{row['title']}\n{row['grid_region_id']}",
    ).add_to(m)

# Legend
legend_html = '<div style="position:fixed;bottom:30px;left:30px;z-index:1000;background:white;padding:10px;border:1px solid #ccc;font-size:12px">'
for region, color in REGION_COLORS.items():
    legend_html += f'<div><span style="color:{color}">&#9679;</span> {region}</div>'
legend_html += "</div>"
m.get_root().html.add_child(folium.Element(legend_html))

m

## Export

Review the map above. If any assignment looks wrong, edit `GRID_MAP` in the cell above and re-run.
Then run this cell to write `data/kek_grid_region_mapping.csv`.

In [ ]:
out = kek[["xid", "slug", "title", "grid_region_id"]].copy()
out.columns = ["kek_id", "slug", "kek_name", "grid_region_id"]

out_path = DATA / "kek_grid_region_mapping.csv"
out.to_csv(out_path, index=False)
print(f"Saved {len(out)} rows → {out_path}")
out